# Please work I need this, my UROP is kind of paperless...

# Imports

In [1]:
import torch
from torch.utils.data import random_split, Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
from tqdm import tqdm
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import numpy as np
import hashlib
from collections import defaultdict
import math

# Constants

In [2]:
batch_size = 1
device = 'cuda'

# Dataset

In [3]:
class ProbDataset(Dataset):
    def __init__(self, X: torch.tensor, Y: torch.tensor, bin_count, bin_width):
        super().__init__()
        self.X = X
        self.Y = Y
        self.bin_count = bin_count
        self.bin_width = bin_width
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.Y[index]

In [4]:
train_dataset = torch.load('data/probabilistic/boiling_point_K_train.pt')
test_dataset = torch.load('data/probabilistic/boiling_point_K_test.pt')

C:\Users\agile\AppData\Local\Temp\ipykernel_14520\2287185075.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load('data/probabilistic/boiling_point

In [5]:
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Prerequisite Calculations

In [6]:
X = train_dataset.X
Y = train_dataset.Y

x_mean = train_dataset.X.mean(dim=0, keepdim=True)
x_std  = train_dataset.X.std(dim=0, keepdim=True)

train_dataset.X = (train_dataset.X - x_mean) / x_std
test_dataset.X = (test_dataset.X - x_mean) / x_std

X_train_stand_min = torch.min(train_dataset.X, dim=0).values.to(device)
X_train_stand_max = torch.max(train_dataset.X, dim=0).values.to(device)

print(X_train_stand_min.shape, X_train_stand_max.shape)

torch.Size([6]) torch.Size([6])


# Model

In [7]:
# class LearnedBayesianRegression(nn.Module):
#     def __init__(self, num_indices, num_bins, min_devs, max_devs):
#         super().__init__()
        
#         self.min_devs = min_devs
#         self.max_devs = max_devs
#         self.num_indices = num_indices
#         self.num_bins = num_bins
        
#         self.W_one = torch.nn.Parameter(torch.randn(1, num_indices, num_bins) * 0.01)
#         self.W_zero = torch.nn.Parameter(torch.randn(1, num_indices, num_bins) * 0.01)
#         self.bias = torch.nn.Parameter(torch.randn(num_bins))
        
#         self.classifier = nn.Sequential(
#             nn.Linear(num_bins, num_bins * 2),
#             nn.ReLU(),
#             nn.Linear(num_bins * 2, num_bins),
#         )

#     def forward(self, devs):
#         denom = (self.max_devs - self.min_devs).clamp(min=1e-6)
#         one_ness = (devs - self.min_devs) / denom
#         one_ness_dimmed = one_ness.unsqueeze(-1)
#         zero_ness_dimmed = 1 - one_ness_dimmed
        
#         logits_iter = one_ness_dimmed * self.W_one + zero_ness_dimmed * self.W_zero
#         reg_logits = self.bias + logits_iter.sum(dim=1)
#         return (reg_logits), self.classifier(reg_logits.detach())

In [8]:
class MIC(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.fc1 = nn.Linear(input_features, 32)
        self.fc2 = nn.Linear(32, 64)
        self.fc3 = nn.Linear(64, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 16)
        self.fc6 = nn.Linear(16, output_features)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = self.fc6(x)
        
        return x
        

In [9]:
mic = MIC(train_dataset.X.shape[1], train_dataset.Y.shape[1])
summary(
    mic,
    train_dataset.X.shape
)

Layer (type:depth-idx)                   Output Shape              Param #
MIC                                      [136, 317]                --
├─Linear: 1-1                            [136, 32]                 224
├─Linear: 1-2                            [136, 64]                 2,112
├─Linear: 1-3                            [136, 128]                8,320
├─Linear: 1-4                            [136, 64]                 8,256
├─Linear: 1-5                            [136, 16]                 1,040
├─Linear: 1-6                            [136, 317]                5,389
Total params: 25,341
Trainable params: 25,341
Non-trainable params: 0
Total mult-adds (M): 3.45
Input size (MB): 0.00
Forward/backward pass size (MB): 0.68
Params size (MB): 0.10
Estimated Total Size (MB): 0.78

In [10]:
# LBR = LearnedBayesianRegression(X_train_stand_min.shape[0], train_dataset.bin_count, X_train_stand_min, X_train_stand_max)
# summary(
#     LBR,
#     train_dataset.X.shape
# )

In [11]:
# def LBRLoss(pos_weight_f_val=8.2):
#     pos_weight = torch.tensor([pos_weight_f_val], device=device)
#     bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#     def loss_func(reg_logits, class_logits, true):
#         binary_targets = (true != 0).float()
#         mask = (true != 0)
        
#         loss_cls = bce(class_logits, binary_targets)
        
#         # safe regression loss
#         if mask.any():
#             pred = torch.sigmoid(reg_logits[mask])
#             loss_reg = ((pred - true[mask]) ** 2).mean()
#         else:
#             loss_reg = torch.tensor(0.0, device=device)
        
#         return loss_cls + loss_reg
        
#     return loss_func

In [12]:
# def MICLoss(median_diffs):
#     eps = 1e-6
    
#     def loss_function(predicted, true):
#         diffs = torch.sqrt((true - predicted)**2 + 1e-3)

#         md = median_diffs.to(diffs.device)
#         scaled = diffs / (md + eps)

#         loss = torch.log1p(scaled).mean()
#         return loss
    
#     return loss_function

# Training

In [13]:
optimizer = torch.optim.Adam(mic.parameters(), lr=1e-3)
criterion = nn.MSELoss()
num_epochs = 900

In [14]:
mic.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for indices, bin_probs in train_loader:
        optimizer.zero_grad()
        indices = indices.to(device)
        bin_probs = bin_probs.to(device)
        
        preds = mic(indices)
        loss = criterion(preds, bin_probs)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch}: loss = {total_loss / len(train_loader)}")

Epoch 0: loss = 0.007268368730900179
Epoch 1: loss = 0.0016231983936449979
Epoch 2: loss = 0.0015842845150473295
Epoch 3: loss = 0.001577462201941457
Epoch 4: loss = 0.0015712896063846797
Epoch 5: loss = 0.001565655024451724
Epoch 6: loss = 0.0015690706005604144
Epoch 7: loss = 0.0015684090051175136
Epoch 8: loss = 0.0015715340815564988
Epoch 9: loss = 0.0015649796221302548
Epoch 10: loss = 0.0015663658136070486
Epoch 11: loss = 0.0015609664772936788
Epoch 12: loss = 0.0015661773487569437
Epoch 13: loss = 0.0015619622167030362
Epoch 14: loss = 0.001560622681876446
Epoch 15: loss = 0.0015641597276365193
Epoch 16: loss = 0.0015579893124060403
Epoch 17: loss = 0.0015583585596580045
Epoch 18: loss = 0.0015583943942842511
Epoch 19: loss = 0.001559437056986259
Epoch 20: loss = 0.0015589484238957725
Epoch 21: loss = 0.0015586319612553138
Epoch 22: loss = 0.0015576835031585207
Epoch 23: loss = 0.0015554944900085341
Epoch 24: loss = 0.0015577698498365114
Epoch 25: loss = 0.0015515960278528783
E

KeyboardInterrupt: 

In [15]:
mic.eval()
with torch.no_grad():
    for indices, bin_probs in test_loader:
        indices = indices.to(device)
        bin_probs = bin_probs.to(device)
        
        preds = mic(indices)
        
        for val_idx in range(preds.shape[1]):
            print(preds[0][val_idx], bin_probs[0][val_idx])

tensor(0.0116, device='cuda:0') tensor(0.5000, device='cuda:0')
tensor(3.4198e-06, device='cuda:0') tensor(0., device='cuda:0')
tensor(0.0002, device='cuda:0') tensor(0., device='cuda:0')
tensor(0.0001, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.0002, device='cuda:0') tensor(0., device='cuda:0')
tensor(-4.5670e-05, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.0002, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.0002, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.0002, device='cuda:0') tensor(0., device='cuda:0')
tensor(0.0001, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.0003, device='cuda:0') tensor(0., device='cuda:0')
tensor(0.0003, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.0004, device='cuda:0') tensor(0., device='cuda:0')
tensor(-2.9296e-05, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.0016, device='cuda:0') tensor(0., device='cuda:0')
tensor(-4.5221e-05, device='cuda:0') tensor(0., device='cuda:0')
tensor(-0.